# 第二章：旋转表示方法 — 交互式可视化

> 台大林沛群教授《机器人学》第二章

本 Notebook 提供交互式演示，重点说明：
1. Fixed Angles vs Euler Angles
2. **左乘 vs 右乘（本章核心）**
3. 齐次变换矩阵
4. Gimbal Lock

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, interact_manual, FloatSlider, Dropdown
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'WenQuanYi Zen Hei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ── 基本旋转矩阵 ──
def Rx(deg):
    t = np.radians(deg)
    return np.array([[1,0,0],[0,np.cos(t),-np.sin(t)],[0,np.sin(t),np.cos(t)]])

def Ry(deg):
    t = np.radians(deg)
    return np.array([[np.cos(t),0,np.sin(t)],[0,1,0],[-np.sin(t),0,np.cos(t)]])

def Rz(deg):
    t = np.radians(deg)
    return np.array([[np.cos(t),-np.sin(t),0],[np.sin(t),np.cos(t),0],[0,0,1]])

def make_T(R, d):
    """构造 4×4 齐次变换矩阵 T = [[R, d], [0,0,0,1]]"""
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3]  = d
    return T

def draw_frame(ax, R, origin=np.zeros(3), scale=1.0, alpha=1.0,
               colors=('r','g','b'), labels=('x','y','z'), lw=2):
    for i in range(3):
        vec = R[:, i] * scale
        ax.quiver(*origin, *vec, color=colors[i], alpha=alpha,
                  arrow_length_ratio=0.15, linewidth=lw)
        if labels:
            ax.text(*(origin + R[:, i]*scale*1.2), labels[i],
                    color=colors[i], fontsize=9, fontweight='bold')

def setup_3d(ax, lim=1.5):
    ax.set_xlim([-lim,lim]); ax.set_ylim([-lim,lim]); ax.set_zlim([-lim,lim])
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_box_aspect([1,1,1])

print('工具函数加载完成 ✓')


工具函数加载完成 ✓


---
## Part 1：三个基本旋转矩阵验证

In [4]:
def check_rotation_matrix(R, name):
    """验证旋转矩阵的数学性质"""
    print(f"\n── {name} ──")
    print(f"R =\n{np.round(R, 4)}")
    print(f"R^T R = I ?  {np.allclose(R.T @ R, np.eye(3))} ✓")
    print(f"det(R) = {np.round(np.linalg.det(R), 6)} (应为 +1)")

for name, R in [('Rx(30°)', Rx(30)), ('Ry(45°)', Ry(45)), ('Rz(60°)', Rz(60))]:
    check_rotation_matrix(R, name)


── Rx(30°) ──
R =
[[ 1.     0.     0.   ]
 [ 0.     0.866 -0.5  ]
 [ 0.     0.5    0.866]]
R^T R = I ?  True ✓
det(R) = 1.0 (应为 +1)

── Ry(45°) ──
R =
[[ 0.7071  0.      0.7071]
 [ 0.      1.      0.    ]
 [-0.7071  0.      0.7071]]
R^T R = I ?  True ✓
det(R) = 1.0 (应为 +1)

── Rz(60°) ──
R =
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]
R^T R = I ?  True ✓
det(R) = 1.0 (应为 +1)


---
## Part 2：Fixed Angles vs Euler Angles（交互式）

**Fixed X-Y-Z**：$R = R_z(\alpha) R_y(\beta) R_x(\gamma)$，参数顺序 $\gamma, \beta, \alpha$

**Euler Z-Y-X**：$R = R_z(\alpha') R_y(\beta') R_x(\gamma')$，参数顺序 $\alpha', \beta', \gamma'$

当 Fixed(γ,β,α) 的参数与 Euler(α',β',γ') 互为逆序时，两者结果相同。

In [ ]:
from IPython.display import display
from ipywidgets import interactive_output, VBox

_alpha = FloatSlider(value=30, min=-180, max=180, step=5, description='α (Z旋转)')
_beta  = FloatSlider(value=45, min=-90,  max=90,  step=5, description='β (Y旋转)')
_gamma = FloatSlider(value=60, min=-180, max=180, step=5, description='γ (X旋转)')

def compare_fixed_euler(alpha, beta, gamma):
    plt.close('all')
    R_fixed = Rz(alpha) @ Ry(beta) @ Rx(gamma)
    R_euler = Rz(gamma) @ Ry(beta) @ Rx(alpha)

    fig = plt.figure(figsize=(12, 4))
    for idx, (R, title) in enumerate([
        (R_fixed, f'Fixed X-Y-Z(γ={gamma:.0f}°,β={beta:.0f}°,α={alpha:.0f}°)'),
        (R_euler, f'Euler Z-Y-X(α={gamma:.0f}°,β={beta:.0f}°,γ={alpha:.0f}°)'),
    ]):
        ax = fig.add_subplot(1, 2, idx+1, projection='3d')
        ax.set_title(title, fontsize=10)
        draw_frame(ax, np.eye(3), scale=0.6, alpha=0.15, labels=None)
        draw_frame(ax, R, scale=1.0)
        setup_3d(ax)

    are_equal = np.allclose(R_fixed, R_euler)
    fig.suptitle(f'两者结果相同？{"√ YES" if are_equal else "× NO"}',
                 fontsize=13, color='green' if are_equal else 'red')
    plt.tight_layout()
    plt.show()

_out2 = interactive_output(compare_fixed_euler, {'alpha': _alpha, 'beta': _beta, 'gamma': _gamma})
display(VBox([_alpha, _beta, _gamma, _out2]))


---
## Part 3：★ 左乘 vs 右乘（核心）

### 规则：
- **左乘** $R_{new} \cdot R$：绕**世界坐标系（固定）**的轴旋转
- **右乘** $R \cdot R_{new}$：绕**本体坐标系（运动）**的轴旋转

In [ ]:
R_funcs = {'Rx': Rx, 'Ry': Ry, 'Rz': Rz}

_init_axis  = Dropdown(options=['Rx','Ry','Rz'], value='Rx', description='初始旋转轴')
_init_angle = FloatSlider(value=45, min=-180, max=180, step=15, description='初始角度')
_new_axis   = Dropdown(options=['Rx','Ry','Rz'], value='Rz', description='新旋转轴')
_new_angle  = FloatSlider(value=90, min=-180, max=180, step=15, description='新旋转角')

def demo_left_vs_right(init_axis, init_angle, new_axis, new_angle):
    plt.close('all')
    R0   = R_funcs[init_axis](init_angle)
    Rnew = R_funcs[new_axis](new_angle)
    R_left  = Rnew @ R0
    R_right = R0 @ Rnew

    fig = plt.figure(figsize=(15, 5))

    ax1 = fig.add_subplot(131, projection='3d')
    ax1.set_title(f'初始姿态\n{init_axis}({init_angle:.0f}°)', fontsize=11)
    draw_frame(ax1, np.eye(3), scale=0.6, alpha=0.2, labels=None)
    draw_frame(ax1, R0, scale=1.0)
    setup_3d(ax1)

    ax2 = fig.add_subplot(132, projection='3d')
    ax2.set_title(f'左乘：{new_axis}({new_angle:.0f}°) · R₀\n→ 绕【世界{new_axis[-1]}轴】旋转', fontsize=11)
    draw_frame(ax2, np.eye(3), scale=0.6, alpha=0.2, labels=None)
    draw_frame(ax2, R0, scale=0.6, alpha=0.4, colors=('salmon','lightgreen','skyblue'), labels=None)
    draw_frame(ax2, R_left, scale=1.0)
    setup_3d(ax2)

    ax3 = fig.add_subplot(133, projection='3d')
    ax3.set_title(f'右乘：R₀ · {new_axis}({new_angle:.0f}°)\n→ 绕【本体{new_axis[-1]}轴】旋转', fontsize=11)
    draw_frame(ax3, np.eye(3), scale=0.6, alpha=0.2, labels=None)
    draw_frame(ax3, R0, scale=0.6, alpha=0.4, colors=('salmon','lightgreen','skyblue'), labels=None)
    draw_frame(ax3, R_right, scale=1.0)
    setup_3d(ax3)

    same = np.allclose(R_left, R_right)
    fig.suptitle(f'左乘 == 右乘? {"YES（初始为 I 时才可能相等）" if same else "NO（旋转不可交换！）"}',
                 color='green' if same else 'red', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"\n左乘结果 =\n{np.round(R_left,3)}")
    print(f"\n右乘结果 =\n{np.round(R_right,3)}")

_out3 = interactive_output(demo_left_vs_right, {
    'init_axis': _init_axis, 'init_angle': _init_angle,
    'new_axis': _new_axis, 'new_angle': _new_angle,
})
display(VBox([_init_axis, _init_angle, _new_axis, _new_angle, _out3]))


---
## Part 4：Gimbal Lock 可视化

In [7]:
_beta2  = FloatSlider(value=90,   min=-90,  max=90,  step=5,  description='β（中间轴）')
_alpha2 = FloatSlider(value=0,   min=-180, max=180, step=10, description='α（第一轴）')
_gamma2 = FloatSlider(value=0,   min=-180, max=180, step=10, description='γ（第三轴）')

def demo_gimbal_lock(beta, alpha, gamma):
    plt.close('all')
    R = Rz(alpha) @ Ry(beta) @ Rx(gamma)
    is_singular = abs(abs(beta) - 90) < 1.0

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    alphas_test = np.linspace(-90, 90, 7)
    cm = plt.cm.RdYlBu(np.linspace(0, 1, len(alphas_test)))
    for a, c in zip(alphas_test, cm):
        R_test = Rz(a) @ Ry(beta) @ Rx(gamma)
        draw_frame(ax, R_test, scale=0.7, alpha=0.6, colors=(c,c,c), labels=None, lw=1)

    draw_frame(ax, R, scale=1.1, alpha=1.0, lw=3)
    setup_3d(ax, lim=1.3)

    status = "GIMBAL LOCK！改变α已无效" if is_singular else "正常（各α对应不同姿态）"
    ax.set_title(f'Z-Y-X Euler Angles: α={alpha:.0f}°, β={beta:.0f}°, γ={gamma:.0f}°\n{status}',
                 color='red' if is_singular else 'green', fontsize=11)
    plt.tight_layout()
    plt.show()

    if is_singular:
        print("当 β=±90° 时，α 和 γ 只有 (α-γ) 的差值有意义，丢失一个自由度！")
        print("这就是 Gimbal Lock 的数学本质。")

_out4 = interactive_output(demo_gimbal_lock, {'beta': _beta2, 'alpha': _alpha2, 'gamma': _gamma2})
display(VBox([_beta2, _alpha2, _gamma2, _out4]))


---
## Part 5：齐次变换矩阵运算

In [8]:
_rot_z = FloatSlider(value=45,  min=-180, max=180, step=15,  description='旋转角(绕Z)')
_tx    = FloatSlider(value=1.0, min=-2,   max=2,   step=0.1, description='平移 tx')
_ty    = FloatSlider(value=0.5, min=-2,   max=2,   step=0.1, description='平移 ty')

def demo_homogeneous(rot_z, tx, ty):
    plt.close('all')
    R = Rz(rot_z)
    d = np.array([tx, ty, 0.0])
    T = make_T(R, d)

    pts = np.array([[0,0,0],[1,0,0],[1,0.2,0],[0.2,0.2,0],[0.2,1,0],[0,1,0],[0,0,0]]).T
    pts_h = np.vstack([pts, np.ones((1, pts.shape[1]))])
    pts_transformed = (T @ pts_h)[:3]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.set_title('XY 平面视图', fontsize=12)
    ax1.plot(*pts[:2], 'b-o', label='原始（坐标系B）', lw=2, ms=6)
    ax1.plot(*pts_transformed[:2], 'r-o', label='变换后（坐标系A）', lw=2, ms=6)
    ax1.annotate('', xy=(tx, ty), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='g', lw=2.5))
    ax1.text(tx/2-0.3, ty/2+0.1, f'平移({tx:.1f},{ty:.1f})', color='g', fontsize=10)
    ax1.set_aspect('equal'); ax1.grid(True, alpha=0.3); ax1.legend(fontsize=10)
    ax1.set_xlim(-2.5, 3.5); ax1.set_ylim(-2, 3)

    ax2.axis('off')
    T_inv = make_T(R.T, -R.T @ d)
    ax2.text(0.05, 0.95,
             f'T =\n{np.round(T,3)}\n\nT⁻¹ =\n{np.round(T_inv,3)}\n\n'
             f'T·T⁻¹ = I ?  {np.allclose(T @ T_inv, np.eye(4))}',
             transform=ax2.transAxes, fontsize=10, va='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.suptitle(f'齐次变换：旋转 {rot_z:.0f}° + 平移({tx:.1f},{ty:.1f})', fontsize=13)
    plt.tight_layout()
    plt.show()

_out5 = interactive_output(demo_homogeneous, {'rot_z': _rot_z, 'tx': _tx, 'ty': _ty})
display(VBox([_rot_z, _tx, _ty, _out5]))


---
## Part 6：批量导出静态图

导出目录会自动解析为与本 Notebook 同级的 `images/`（即 `Robotics_NTU/images`）。

运行下面代码单元后，执行 `export_all_chapter02_figures()` 可一键生成：
- `ch02_fixed_vs_euler.png`
- `ch02_left_vs_right.png`
- `ch02_homogeneous_transform.png`
- `ch02_gimbal_lock.png`

In [9]:
from pathlib import Path

def _resolve_output_dir():
    cwd = Path.cwd()
    for base in (cwd, cwd / "Robotics_NTU"):
        if (base / "Chapter02_visualization.ipynb").exists():
            out = base / "images"
            out.mkdir(parents=True, exist_ok=True)
            return out
    out = cwd / "images"
    out.mkdir(parents=True, exist_ok=True)
    return out

OUTPUT_DIR = _resolve_output_dir()
print(f"Export directory: {OUTPUT_DIR}")


def export_fixed_vs_euler_png(alpha=30, beta=45, gamma=60):
    R_fixed = Rz(alpha) @ Ry(beta) @ Rx(gamma)
    R_euler = Rz(alpha) @ Ry(beta) @ Rx(gamma)

    fig = plt.figure(figsize=(14, 6))
    fig.suptitle(
        "Fixed Angles vs Euler Angles\n"
        f"Fixed X-Y-Z(Roll={gamma} deg, Pitch={beta} deg, Yaw={alpha} deg) = "
        f"Euler Z-Y-X(a={alpha} deg, b={beta} deg, g={gamma} deg)",
        fontsize=12,
        fontweight="bold",
    )

    ax1 = fig.add_subplot(131, projection="3d")
    ax1.set_title("Fixed Angles\n(world axes, pre-multiply)", pad=8)
    draw_frame(ax1, np.eye(3), scale=0.6, alpha=0.15, labels=None)
    r1 = Rx(gamma)
    r2 = Ry(beta) @ r1
    r3 = Rz(alpha) @ r2
    draw_frame(ax1, r1, scale=0.5, alpha=0.4, colors=("salmon", "lightgreen", "skyblue"), labels=None)
    draw_frame(ax1, r2, scale=0.7, alpha=0.6, colors=("orangered", "mediumseagreen", "cornflowerblue"), labels=None)
    draw_frame(ax1, r3, scale=1.0, alpha=1.0)
    setup_3d(ax1)

    ax2 = fig.add_subplot(132, projection="3d")
    ax2.set_title("Euler Angles\n(body axes, post-multiply)", pad=8)
    draw_frame(ax2, np.eye(3), scale=0.6, alpha=0.15, labels=None)
    e1 = Rz(alpha)
    e2 = e1 @ Ry(beta)
    e3 = e2 @ Rx(gamma)
    draw_frame(ax2, e1, scale=0.5, alpha=0.4, colors=("salmon", "lightgreen", "skyblue"), labels=None)
    draw_frame(ax2, e2, scale=0.7, alpha=0.6, colors=("orangered", "mediumseagreen", "cornflowerblue"), labels=None)
    draw_frame(ax2, e3, scale=1.0, alpha=1.0)
    setup_3d(ax2)

    ax3 = fig.add_subplot(133, projection="3d")
    ax3.set_title("Final pose comparison", pad=8)
    draw_frame(ax3, np.eye(3), scale=0.5, alpha=0.2, labels=None)
    draw_frame(ax3, R_fixed, scale=1.0, alpha=0.9, colors=("r", "g", "b"), labels=("Fx", "Fy", "Fz"))
    for i, c in enumerate(("darkred", "darkgreen", "navy")):
        vec = R_euler[:, i]
        ax3.quiver(0, 0, 0, *vec, color=c, alpha=0.4, arrow_length_ratio=0.1, linewidth=4, linestyle="dashed")
    setup_3d(ax3)

    fig.tight_layout()
    path = OUTPUT_DIR / "ch02_fixed_vs_euler.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def export_left_vs_right_png():
    r0 = Rx(45)
    rz90 = Rz(90)
    r_left = rz90 @ r0
    r_right = r0 @ rz90

    fig = plt.figure(figsize=(15, 5))
    fig.suptitle("Left vs Right Multiply: R0 = Rx(45 deg), then apply Rz(90 deg)", fontsize=12, fontweight="bold")

    ax1 = fig.add_subplot(131, projection="3d")
    ax1.set_title("Initial pose R0")
    draw_frame(ax1, np.eye(3), scale=0.8, alpha=0.2, colors=("salmon", "lightgreen", "skyblue"), labels=None)
    draw_frame(ax1, r0, scale=1.0)
    body_z = r0[:, 2]
    ax1.quiver(0, 0, 0, *(body_z * 1.3), color="navy", linewidth=3, alpha=0.5, arrow_length_ratio=0.1, linestyle="dashed", label="body Z")
    ax1.legend(fontsize=8)
    setup_3d(ax1)

    ax2 = fig.add_subplot(132, projection="3d")
    ax2.set_title("Pre-multiply: Rz * R0\nrotate around WORLD Z")
    draw_frame(ax2, np.eye(3), scale=0.8, alpha=0.2, colors=("salmon", "lightgreen", "skyblue"), labels=None)
    draw_frame(ax2, r0, scale=0.7, alpha=0.4, colors=("salmon", "lightgreen", "skyblue"), labels=None)
    draw_frame(ax2, r_left, scale=1.0)
    ax2.quiver(0, 0, -1.2, 0, 0, 2.4, color="blue", linewidth=2, alpha=0.6, arrow_length_ratio=0.07, label="WORLD Z")
    ax2.legend(fontsize=8)
    setup_3d(ax2)

    ax3 = fig.add_subplot(133, projection="3d")
    ax3.set_title("Post-multiply: R0 * Rz\nrotate around BODY Z")
    draw_frame(ax3, np.eye(3), scale=0.8, alpha=0.2, colors=("salmon", "lightgreen", "skyblue"), labels=None)
    draw_frame(ax3, r0, scale=0.7, alpha=0.4, colors=("salmon", "lightgreen", "skyblue"), labels=None)
    draw_frame(ax3, r_right, scale=1.0)
    body_z = r0[:, 2]
    ax3.quiver(*(-body_z * 1.2), *(body_z * 2.4), color="darkblue", linewidth=2, alpha=0.6, arrow_length_ratio=0.07, label="BODY Z")
    ax3.legend(fontsize=8)
    setup_3d(ax3)

    fig.tight_layout()
    path = OUTPUT_DIR / "ch02_left_vs_right.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def export_homogeneous_transform_png():
    r = Rz(45)
    d = np.array([1.0, 0.5, 0.0])
    t = make_T(r, d)

    pts_b = np.array([
        [0, 0, 0], [1, 0, 0], [1, 0.2, 0], [0.2, 0.2, 0],
        [0.2, 1, 0], [0, 1, 0], [0, 0, 0],
    ]).T
    pts_b_h = np.vstack([pts_b, np.ones((1, pts_b.shape[1]))])
    pts_a_h = t @ pts_b_h
    pts_a = pts_a_h[:3]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Homogeneous Transform: rotate 45 deg + translate (1, 0.5, 0)", fontsize=12, fontweight="bold")

    ax = axes[0]
    ax.set_title("Top view (XY)")
    ax.plot(*pts_b[:2], "b-o", label="Original in frame B", lw=2, markersize=5)
    ax.plot(*pts_a[:2], "r-o", label="Transformed in frame A", lw=2, markersize=5)
    ax.annotate("", xy=(d[0], d[1]), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="green", lw=2))
    ax.text(d[0] / 2, d[1] / 2 + 0.1, "translation", color="green", fontsize=10)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    ax2 = axes[1]
    ax2.axis("off")
    matrix_text = (
        f"T =\n\n"
        f"  [[{r[0,0]:6.3f}, {r[0,1]:6.3f}, {r[0,2]:6.3f}, {d[0]:6.3f}],\n"
        f"   [{r[1,0]:6.3f}, {r[1,1]:6.3f}, {r[1,2]:6.3f}, {d[1]:6.3f}],\n"
        f"   [{r[2,0]:6.3f}, {r[2,1]:6.3f}, {r[2,2]:6.3f}, {d[2]:6.3f}],\n"
        f"   [ 0.000,  0.000,  0.000,  1.000]]"
    )
    ax2.text(0.05, 0.95, matrix_text, transform=ax2.transAxes, fontsize=10, va="top", fontfamily="monospace", bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

    fig.tight_layout()
    path = OUTPUT_DIR / "ch02_homogeneous_transform.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def export_gimbal_lock_png():
    fig = plt.figure(figsize=(15, 5))
    fig.suptitle("Gimbal Lock in Z-Y-X Euler Angles at beta = 90 deg", fontsize=12, fontweight="bold")

    beta = 90
    vals = [0, 30, 60, 90]
    colors_map = plt.cm.cool(np.linspace(0, 1, len(vals)))

    ax1 = fig.add_subplot(131, projection="3d")
    ax1.set_title("beta=90, vary alpha (gamma=0)")
    for a, c in zip(vals, colors_map):
        r = Rz(a) @ Ry(beta) @ Rx(0)
        draw_frame(ax1, r, scale=0.7, alpha=0.8, colors=(c, c, c), labels=None)
    setup_3d(ax1)

    ax2 = fig.add_subplot(132, projection="3d")
    ax2.set_title("beta=90, vary gamma (alpha=0)")
    for g, c in zip(vals, colors_map):
        r = Rz(0) @ Ry(beta) @ Rx(g)
        draw_frame(ax2, r, scale=0.7, alpha=0.8, colors=(c, c, c), labels=None)
    setup_3d(ax2)

    ax3 = fig.add_subplot(133, projection="3d")
    ax3.set_title("beta=45 (non-singular)")
    beta2 = 45
    for a, c in zip(vals, colors_map):
        r = Rz(a) @ Ry(beta2) @ Rx(0)
        draw_frame(ax3, r, scale=0.7, alpha=0.8, colors=(c, c, c), labels=None)
    setup_3d(ax3)

    fig.tight_layout()
    path = OUTPUT_DIR / "ch02_gimbal_lock.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def export_all_chapter02_figures():
    paths = [
        export_fixed_vs_euler_png(),
        export_left_vs_right_png(),
        export_homogeneous_transform_png(),
        export_gimbal_lock_png(),
    ]
    print("Exported files:")
    for p in paths:
        print(f"- {p}")
    return paths


# Run this to export all figures:
# export_all_chapter02_figures()

Export directory: d:\EI-Beginner\Robotics_NTU\images


---
## 总结

| 概念 | 一句话总结 |
|------|----------|
| Fixed Angles | 绕固定世界轴，矩阵**左乘**累积 |
| Euler Angles | 绕运动本体轴，矩阵**右乘**累积 |
| 左乘 $R_{new}\cdot R$ | 新旋转相对**世界坐标系** |
| 右乘 $R \cdot R_{new}$ | 新旋转相对**本体坐标系** |
| X-Y-Z Fixed = Z-Y-X Euler | 参数互为逆序时等价 |
| Gimbal Lock | Euler中间轴 ±90° 时丢失一个自由度 |